# 09 — Learning Curves

**Difficulty:** ⭐⭐⭐ &nbsp;|&nbsp; **Estimated time:** 2 hours &nbsp;|&nbsp; **Week 8: Regularization & Optimization**

---

## Why This Matters

You've built a model, tuned some hyperparameters, and you're wondering: *should I collect more data, or is my model architecture the problem?* Learning curves answer that question **before** you spend weeks and money gathering data.

They are the X-ray of machine learning diagnosis — a single plot tells you whether your patient (model) is suffering from **underfitting (high bias)** or **overfitting (high variance)**.

---

## The Analogy: A Student Practising Maths

Imagine a student preparing for an exam by working through practice problems:

- After **10 problems**, their mock-exam score jumps from 40 → 65. Huge improvement!
- After **100 problems**, it climbs to 78. Still good.
- After **1 000 problems**, it nudges to 80.
- After **2 000 problems**, it barely moves to 81.

Plot *"number of problems solved"* on the x-axis and *"mock-exam score"* on the y-axis — that's a **learning curve**. At some point the student has **plateaued**: more practice barely helps. That's the signal to change *strategy* (harder problems, a different textbook, a tutor) rather than just doing *more of the same*.

Your ML model is the student. Training samples are practice problems. The learning curve tells you when more data helps and when it doesn't.

---

## Plain-English Concept

A **learning curve** trains your model on progressively larger subsets of the training data — say 10 %, 20 %, …, 100 % — and records:

1. **Training score** (or error) at each size
2. **Validation score** (or error) at each size

### What the shapes tell you

| Pattern | Training error | Validation error | Diagnosis | Action |
|---|---|---|---|---|
| High bias | High and flat | High and flat, converging to training | Underfitting | More complex model, more features, less regularisation |
| High variance | Low | High, large gap | Overfitting | More data, simpler model, more regularisation |
| Well-fit | Low | Low, small gap | Good fit | Accept model, maybe collect a bit more data |

### The key insight

- **High bias:** Both curves plateau at a **high error**. More data barely helps — the model can't learn the pattern even with infinite data.
- **High variance:** Large **gap** between training (low) and validation (high) error. More data *does* help — the gap narrows as training size grows.

---

## The Mathematics

For a model trained on $m$ samples from dataset $D$:

$$\text{Training error}(m) = \frac{1}{m}\sum_{i=1}^{m}\mathcal{L}(y_i,\hat{y}_i)$$

$$\text{Validation error}(m) = \frac{1}{|D_{val}|}\sum_{j \in D_{val}}\mathcal{L}(y_j, \hat{y}_j(\theta_m))$$

As $m \to \infty$:
- Training error **increases** (harder to perfectly fit more diverse data)
- Validation error **decreases** (model generalises better)
- The two curves **converge** — the gap closes

The **bias-variance trade-off** determines *where* they converge:
- High-bias model: both converge to a high error floor
- High-variance model: they converge slowly; gap remains large at practical data sizes

---

## sklearn API

```python
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    estimator  = model,
    X          = X,
    y          = y,
    train_sizes= np.linspace(0.1, 1.0, 10),   # 10 %, 20 %, …, 100 %
    cv         = 5,                             # 5-fold cross-validation
    scoring    = 'neg_mean_squared_error',
    n_jobs     = -1
)
# train_scores shape: (n_sizes, n_cv_folds) — take mean across axis=1
```

## Section 1 — Setup & House-Price Dataset

We simulate a California-style house-price dataset with 1 000 samples and 20 features. The target is house price in $100 k units.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge, ElasticNetCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import learning_curve, KFold
from sklearn.metrics import mean_squared_error

# ── Reproducibility ──────────────────────────────────────────────────────────
np.random.seed(42)

# ── Matplotlib style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
sns.set_palette('tab10')

# ── Generate synthetic house-price dataset ───────────────────────────────────
N_SAMPLES  = 1000
N_FEATURES = 20

# Features: square footage, rooms, age, location proxies, economic indicators …
X_raw = np.random.randn(N_SAMPLES, N_FEATURES)

# True relationship: price depends on first 8 features + mild interaction
true_coef = np.zeros(N_FEATURES)
true_coef[:8] = np.array([3.0, 1.5, -0.8, 2.2, -1.1, 0.6, 1.9, -0.4])

noise     = np.random.normal(0, 1.5, N_SAMPLES)
y_raw     = X_raw @ true_coef + 0.3 * X_raw[:, 0] * X_raw[:, 1] + noise

# ── Scale to feel like house prices ($100 k) ─────────────────────────────────
y = (y_raw - y_raw.mean()) / y_raw.std() * 2 + 5   # mean ≈ 5, std ≈ 2
X = X_raw.copy()

feature_names = (
    ['sqft', 'rooms', 'age', 'loc_score', 'income', 'school_dist',
     'crime_rate', 'transit'] +
    [f'noise_{i}' for i in range(1, N_FEATURES - 7)]
)
df = pd.DataFrame(X, columns=feature_names)
df['price_100k'] = y

print(f"Dataset shape : {df.shape}")
print(f"Price range   : ${df['price_100k'].min():.2f}–${df['price_100k'].max():.2f} (×100 k)")
df[['sqft', 'rooms', 'age', 'price_100k']].describe().round(2)

## Section 2 — Three Models Representing Three Regimes

We build three polynomial regression pipelines using only two features (`sqft`, `rooms`) so we can use standard polynomial expansion:

| Model | Polynomial degree | Expected behaviour |
|---|---|---|
| `poly1` | 1 (linear) | High bias — too simple for a nonlinear relationship |
| `poly10` | 10 | High variance — memorises training data |
| `poly4` | 4 | Good fit — captures mild nonlinearity without over-fitting |

In [ ]:
from sklearn.linear_model import LinearRegression

# Use only two features for polynomial expansion to keep it tractable
X2 = X[:, :2]   # sqft and rooms

def make_poly_pipeline(degree):
    """StandardScaler → PolynomialFeatures → LinearRegression pipeline."""
    return Pipeline([
        ('scaler', StandardScaler()),
        ('poly',   PolynomialFeatures(degree=degree, include_bias=False)),
        ('lr',     LinearRegression()),
    ])

# Three models representing the three bias-variance regimes
models = {
    'Degree 1  (High Bias)':    make_poly_pipeline(1),
    'Degree 4  (Good Fit)':     make_poly_pipeline(4),
    'Degree 10 (High Variance)': make_poly_pipeline(10),
}

# Quick sanity check: fit each on the full dataset and report train MSE
for name, model in models.items():
    model.fit(X2, y)
    train_mse = mean_squared_error(y, model.predict(X2))
    print(f"{name:35s}  Train MSE = {train_mse:.4f}")

print("\nNote: degree-10 memorises the data almost perfectly (very low train MSE).")
print("The learning curves in the next section will show how badly it generalises.")

## Section 3 — Learning Curves for All Three Models

Each subplot shows training MSE (solid) and validation MSE (dashed) versus the number of training samples used. The **gap** between the curves is the visual signature of variance.

In [ ]:
def get_learning_curve_data(model, X, y, train_sizes_frac, cv=5):
    """Wrapper around sklearn's learning_curve that returns mean ± std MSE."""
    sizes, tr_scores, val_scores = learning_curve(
        model, X, y,
        train_sizes=train_sizes_frac,
        cv=cv,
        scoring='neg_mean_squared_error',
        shuffle=True,
        random_state=42,
        n_jobs=-1,
    )
    # neg_MSE → positive MSE
    tr_mean  = -tr_scores.mean(axis=1)
    tr_std   = tr_scores.std(axis=1)
    val_mean = -val_scores.mean(axis=1)
    val_std  = val_scores.std(axis=1)
    return sizes, tr_mean, tr_std, val_mean, val_std


TRAIN_FRACS = np.linspace(0.08, 1.0, 12)

# Re-initialise models (unfitted) before calling learning_curve
models_fresh = {
    'Degree 1\n(High Bias)':     make_poly_pipeline(1),
    'Degree 4\n(Good Fit)':      make_poly_pipeline(4),
    'Degree 10\n(High Variance)': make_poly_pipeline(10),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
colours = ['steelblue', 'tomato']

for ax, (name, model) in zip(axes, models_fresh.items()):
    sizes, tr_m, tr_s, val_m, val_s = get_learning_curve_data(model, X2, y, TRAIN_FRACS)

    ax.plot(sizes, tr_m,  color=colours[0], lw=2, label='Training MSE')
    ax.fill_between(sizes, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color=colours[0])

    ax.plot(sizes, val_m, color=colours[1], lw=2, ls='--', label='Validation MSE')
    ax.fill_between(sizes, val_m - val_s, val_m + val_s, alpha=0.15, color=colours[1])

    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Training set size')
    ax.set_ylabel('MSE')
    ax.legend(fontsize=9)
    ax.set_ylim(bottom=0)

fig.suptitle('Learning Curves — Three Bias-Variance Regimes\n(House Price Prediction)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Observation guide:")
print("  Left  : Both curves are HIGH and close → high bias (underfitting)")
print("  Middle: Both curves are LOW and converging → good fit")
print("  Right : Large GAP between curves → high variance (overfitting)")

## Section 4 — Learning Curve From Scratch (No sklearn)

Understanding the algorithm behind `learning_curve` helps you customise it. Here we loop over training-set sizes manually, run K-fold CV at each size, and accumulate MSE.

In [ ]:
from sklearn.base import clone

def learning_curve_scratch(estimator, X, y, train_sizes, n_splits=5, random_state=42):
    """
    Manual learning-curve implementation.
    Returns arrays: sizes, train_mse_mean, val_mse_mean
    """
    rng   = np.random.RandomState(random_state)
    kf    = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    all_train_mse = []
    all_val_mse   = []
    actual_sizes  = []

    for frac in train_sizes:
        fold_train_mse = []
        fold_val_mse   = []

        for train_idx, val_idx in kf.split(X):
            # Subsample the training fold to 'frac' of its size
            n_sub   = max(2, int(len(train_idx) * frac))
            sub_idx = rng.choice(train_idx, size=n_sub, replace=False)

            X_tr, y_tr = X[sub_idx], y[sub_idx]
            X_v,  y_v  = X[val_idx], y[val_idx]

            # Clone ensures a fresh (unfitted) estimator each fold
            est = clone(estimator)
            est.fit(X_tr, y_tr)

            fold_train_mse.append(mean_squared_error(y_tr, est.predict(X_tr)))
            fold_val_mse.append(mean_squared_error(y_v,  est.predict(X_v)))

        all_train_mse.append(np.mean(fold_train_mse))
        all_val_mse.append(np.mean(fold_val_mse))
        actual_sizes.append(int(len(X) * (n_splits - 1) / n_splits * frac))

    return (np.array(actual_sizes),
            np.array(all_train_mse),
            np.array(all_val_mse))


# Test on the degree-4 model (good fit)
model_d4 = make_poly_pipeline(4)
fracs     = np.linspace(0.05, 1.0, 12)

sizes_s, tr_mse_s, val_mse_s = learning_curve_scratch(model_d4, X2, y, fracs)

plt.figure(figsize=(8, 4))
plt.plot(sizes_s, tr_mse_s,  'o-', color='steelblue', label='Training MSE (scratch)')
plt.plot(sizes_s, val_mse_s, 's--', color='tomato',   label='Validation MSE (scratch)')
plt.xlabel('Training set size')
plt.ylabel('MSE')
plt.title('Learning Curve — Degree-4 Model, Built From Scratch')
plt.legend()
plt.ylim(bottom=0)
plt.tight_layout()
plt.show()

print(f"Final training MSE   : {tr_mse_s[-1]:.4f}")
print(f"Final validation MSE : {val_mse_s[-1]:.4f}")
print(f"Gap at full data     : {val_mse_s[-1] - tr_mse_s[-1]:.4f}")

## Section 5 — Annotated Diagnostic Zones

We add shaded zones directly on the learning curves to label the diagnostic regions: **underfitting zone** (where both errors are high), **overfitting zone** (where the gap is large), and the **convergence zone** (where the curves come together).

In [ ]:
# Focus on the high-variance model to show the overfitting zone clearly
model_d10 = make_poly_pipeline(10)
sizes_hv, tr_hv, tr_hv_s, val_hv, val_hv_s = get_learning_curve_data(
    model_d10, X2, y, TRAIN_FRACS
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: high-bias annotations ──────────────────────────────────────────────
model_d1 = make_poly_pipeline(1)
sizes_hb, tr_hb, tr_hb_s, val_hb, val_hb_s = get_learning_curve_data(
    model_d1, X2, y, TRAIN_FRACS
)
ax = axes[0]
ax.plot(sizes_hb, tr_hb,  color='steelblue', lw=2, label='Training MSE')
ax.plot(sizes_hb, val_hb, color='tomato',    lw=2, ls='--', label='Validation MSE')
ax.axhspan(min(val_hb) * 0.9, max(val_hb) * 1.05,
           alpha=0.08, color='orange', label='Underfitting zone')
ax.annotate('Underfitting zone\n(both curves plateau HIGH)',
            xy=(sizes_hb[3], val_hb[3]),
            xytext=(sizes_hb[5], val_hb[5] + 0.3),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=9, color='darkorange')
ax.set_title('Degree 1 — High Bias (Underfitting)', fontweight='bold')
ax.set_xlabel('Training set size'); ax.set_ylabel('MSE')
ax.legend(fontsize=9); ax.set_ylim(bottom=0)

# ── Right: high-variance annotations ─────────────────────────────────────────
ax = axes[1]
ax.plot(sizes_hv, tr_hv,  color='steelblue', lw=2, label='Training MSE')
ax.plot(sizes_hv, val_hv, color='tomato',    lw=2, ls='--', label='Validation MSE')

# Shade the gap between training and validation curves
ax.fill_between(sizes_hv, tr_hv, val_hv, alpha=0.15, color='red',
                label='Overfitting gap')
ax.annotate('Overfitting zone\n(large gap between curves)',
            xy=(sizes_hv[4], (tr_hv[4] + val_hv[4]) / 2),
            xytext=(sizes_hv[2], val_hv[4] + 1.5),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=9, color='crimson')
ax.annotate('More data helps\n(gap narrows right)',
            xy=(sizes_hv[-2], (tr_hv[-2] + val_hv[-2]) / 2),
            xytext=(sizes_hv[-4], val_hv[-1] + 1.0),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=9, color='darkgreen')
ax.set_title('Degree 10 — High Variance (Overfitting)', fontweight='bold')
ax.set_xlabel('Training set size'); ax.set_ylabel('MSE')
ax.legend(fontsize=9); ax.set_ylim(bottom=0)

plt.suptitle('Annotated Learning Curve Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 6 — Regularisation Effect on Learning Curves

We apply Ridge regression with three different alpha (regularisation strength) values to the degree-10 model:

| Alpha | Effect |
|---|---|
| 0.001 | Barely any regularisation → still overfits |
| 1.0 | Balanced regularisation → good fit |
| 1000 | Heavy regularisation → model too constrained → underfits |

In [ ]:
def make_ridge_pipeline(degree, alpha):
    """Pipeline: StandardScaler → PolynomialFeatures → Ridge(alpha)."""
    return Pipeline([
        ('scaler', StandardScaler()),
        ('poly',   PolynomialFeatures(degree=degree, include_bias=False)),
        ('ridge',  Ridge(alpha=alpha)),
    ])

alpha_configs = [
    (0.001, 'α=0.001 (near-zero regularisation → overfit)'),
    (1.0,   'α=1.0   (balanced regularisation → good fit)'),
    (1000,  'α=1000  (heavy regularisation → underfit)'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
colors_tr  = ['steelblue', 'seagreen', 'purple']
colors_val = ['tomato',    'darkorange', 'mediumpurple']

for ax, (alpha, label), ct, cv_col in zip(axes, alpha_configs, colors_tr, colors_val):
    model_r = make_ridge_pipeline(degree=10, alpha=alpha)
    sz, tr_m, tr_s, val_m, val_s = get_learning_curve_data(model_r, X2, y, TRAIN_FRACS)

    ax.plot(sz, tr_m,  color=ct,     lw=2,      label='Training MSE')
    ax.plot(sz, val_m, color=cv_col, lw=2, ls='--', label='Validation MSE')
    ax.fill_between(sz, tr_m, val_m, alpha=0.10, color='grey')
    ax.fill_between(sz, tr_m - tr_s, tr_m + tr_s, alpha=0.10, color=ct)
    ax.fill_between(sz, val_m - val_s, val_m + val_s, alpha=0.10, color=cv_col)

    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel('Training set size')
    ax.set_ylabel('MSE')
    ax.legend(fontsize=9)
    ax.set_ylim(bottom=0)

    gap = val_m[-1] - tr_m[-1]
    ax.text(0.05, 0.95, f'Final gap: {gap:.3f}',
            transform=ax.transAxes, fontsize=9, va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Ridge Regularisation Effect on Learning Curves\n'
             '(Degree-10 Polynomial, House Price Data)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Section 7 — Estimating How Much More Data Is Needed

When the learning curve shows a **high-variance** pattern, more data helps — but how much? We can extrapolate the validation-error curve and find the sample size where the gap closes to a target threshold (e.g., 10 %).

In [ ]:
# ── Extrapolate with a power-law fit: val_error ≈ a * n^(-b) + c ─────────────
from scipy.optimize import curve_fit

def power_law(n, a, b, c):
    return a * np.power(n, -b) + c

# Use the degree-10 (high-variance) model curves
sizes_hv_ext = sizes_hv.astype(float)
val_hv_clean = np.clip(val_hv, 1e-6, None)

try:
    popt, _ = curve_fit(power_law, sizes_hv_ext, val_hv_clean,
                        p0=[10, 0.3, val_hv_clean[-1]], maxfev=10000)
    a_fit, b_fit, c_fit = popt
except RuntimeError:
    # Fallback: simple linear extrapolation in log space
    a_fit, b_fit, c_fit = 5.0, 0.3, val_hv_clean[-1]

# Target: validation error within 10 % of the training error at full data
train_err_full = tr_hv[-1]
target_val_err = train_err_full * 1.10     # 10 % gap threshold

n_extended = np.linspace(sizes_hv_ext[0], sizes_hv_ext[-1] * 8, 500)
val_extrap  = power_law(n_extended, a_fit, b_fit, c_fit)

# Find first n where extrapolated val error ≤ target
crossings = n_extended[val_extrap <= target_val_err]
n_needed  = crossings[0] if len(crossings) > 0 else None

# ── Plot ─────────────────────────────────────────────────────────────────────
plt.figure(figsize=(9, 5))
plt.plot(sizes_hv, tr_hv,   'o-', color='steelblue', lw=2, label='Training MSE (actual)')
plt.plot(sizes_hv, val_hv,  's--', color='tomato',   lw=2, label='Validation MSE (actual)')
plt.plot(n_extended, val_extrap, ':', color='darkorange', lw=2, label='Validation MSE (extrapolated)')
plt.axhline(target_val_err, color='green', ls='--', lw=1.5,
            label=f'Target val MSE = {target_val_err:.3f} (10 % above train)')

if n_needed:
    plt.axvline(n_needed, color='purple', ls='--', lw=1.5,
                label=f'Estimated n needed ≈ {int(n_needed)}')
    plt.text(n_needed + 5, max(val_hv) * 0.6, f'≈ {int(n_needed)} samples',
             color='purple', fontsize=10)

plt.axvline(sizes_hv[-1], color='grey', ls=':', alpha=0.7, label='Current data boundary')
plt.xlabel('Training set size')
plt.ylabel('MSE')
plt.title('Sample-Size Estimation via Learning-Curve Extrapolation\n'
          '(Degree-10 Model, House Price Data)')
plt.legend(fontsize=9)
plt.ylim(bottom=0)
plt.tight_layout()
plt.show()

print(f"Current training size   : {int(sizes_hv[-1])} samples")
print(f"Target validation MSE   : {target_val_err:.4f}")
if n_needed:
    print(f"Estimated samples needed: ~{int(n_needed)} ({int(n_needed - sizes_hv[-1])} more)")
else:
    print("Extrapolation suggests the gap may not close with this model — consider simplifying.")

## Section 8 — Real Pipeline: StandardScaler + ElasticNetCV

A production-grade sklearn pipeline uses cross-validated regularisation selection. We verify it shows **stable convergence** — both curves descend and meet without a large persistent gap.

In [ ]:
# ElasticNetCV auto-selects the best alpha and l1_ratio via cross-validation
enet_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('enet',   ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.9, 1.0],
        alphas=np.logspace(-3, 2, 30),
        cv=5,
        max_iter=5000,
        random_state=42,
    ))
])

# Use all 20 features for this pipeline
TRAIN_FRACS_ENET = np.linspace(0.08, 1.0, 12)

sizes_e, tr_e, tr_es, val_e, val_es = get_learning_curve_data(
    enet_pipeline, X, y, TRAIN_FRACS_ENET, cv=5
)

plt.figure(figsize=(9, 5))
plt.plot(sizes_e, tr_e,  'o-', color='seagreen',    lw=2.5, label='Training MSE')
plt.fill_between(sizes_e, tr_e - tr_es, tr_e + tr_es, alpha=0.15, color='seagreen')

plt.plot(sizes_e, val_e, 's--', color='darkorange', lw=2.5, label='Validation MSE')
plt.fill_between(sizes_e, val_e - val_es, val_e + val_es, alpha=0.15, color='darkorange')

# Mark convergence zone
conv_idx  = np.argmin(np.abs(val_e - tr_e))   # closest point
plt.annotate('Curves converging —\ngood generalisation',
             xy=(sizes_e[conv_idx], (val_e[conv_idx] + tr_e[conv_idx]) / 2),
             xytext=(sizes_e[conv_idx] - 80, val_e[conv_idx] + 0.5),
             arrowprops=dict(arrowstyle='->', color='black'),
             fontsize=9)

plt.xlabel('Training set size')
plt.ylabel('MSE')
plt.title('Learning Curve — StandardScaler + ElasticNetCV Pipeline\n'
          '(All 20 Features, House Price Data)')
plt.legend()
plt.ylim(bottom=0)
plt.tight_layout()
plt.show()

# Fit full pipeline to report selected hyperparameters
enet_pipeline.fit(X, y)
enet_model = enet_pipeline.named_steps['enet']
print(f"Selected alpha   : {enet_model.alpha_:.5f}")
print(f"Selected l1_ratio: {enet_model.l1_ratio_:.2f}")
print(f"Non-zero coefs   : {np.sum(enet_model.coef_ != 0)} / {len(enet_model.coef_)}")
print(f"Final gap (val - train MSE): {val_e[-1] - tr_e[-1]:.4f}")

## Section 9 — Summary Dashboard

A single figure comparing all models' final (full-data) training vs validation MSE — a quick performance comparison table.

In [ ]:
# Gather final-point metrics for all models studied
summary_data = {
    'Degree 1 (High Bias)':      (tr_hb[-1], val_hb[-1]),
    'Degree 4 (Good Fit)':       (tr_mse_s[-1], val_mse_s[-1]),
    'Degree 10 (High Variance)': (tr_hv[-1], val_hv[-1]),
    'Ridge α=0.001':             None,   # placeholder — computed below
    'Ridge α=1.0':               None,
    'Ridge α=1000':              None,
    'ElasticNetCV Pipeline':     (tr_e[-1], val_e[-1]),
}

# Fill Ridge values from earlier learning curve runs
for alpha, label in alpha_configs:
    model_r = make_ridge_pipeline(degree=10, alpha=alpha)
    sz, tr_m, _, val_m, _ = get_learning_curve_data(model_r, X2, y, TRAIN_FRACS)
    key = f'Ridge α={alpha}'
    summary_data[key] = (tr_m[-1], val_m[-1])

# Build summary DataFrame (drop None entries)
rows = [(k, v[0], v[1], v[1] - v[0])
        for k, v in summary_data.items() if v is not None]
df_sum = pd.DataFrame(rows, columns=['Model', 'Train MSE', 'Val MSE', 'Gap'])
df_sum = df_sum.sort_values('Val MSE').reset_index(drop=True)

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
x     = np.arange(len(df_sum))
width = 0.35

bars1 = ax.bar(x - width/2, df_sum['Train MSE'], width, label='Train MSE',
               color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, df_sum['Val MSE'],   width, label='Val MSE',
               color='tomato',    alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(df_sum['Model'], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('MSE')
ax.set_title('Final Train vs Validation MSE — All Models Compared', fontweight='bold')
ax.legend()

# Annotate gap on each bar pair
for xi, gap in zip(x, df_sum['Gap']):
    ax.text(xi, max(df_sum['Val MSE']) * 1.01, f'Δ{gap:.2f}',
            ha='center', fontsize=7.5, color='darkred')

plt.tight_layout()
plt.show()

print("\nSummary Table:")
print(df_sum.to_string(index=False, float_format='{:.4f}'.format))

## Section 10 — Key Takeaways

| Learning curve signal | Root cause | Fix |
|---|---|---|
| Both curves HIGH and flat | High bias (underfitting) | More complex model, more features, less regularisation |
| Large gap, val >> train | High variance (overfitting) | More data, simpler model, more regularisation |
| Both curves LOW and converging | Well-fit | Accept model; marginal gains from more data |
| Val curve still falling at max n | Data-limited | Collect more data — it will help |
| Val curve flat but far above train | Model-limited | Architecture change needed — data won't fix this |

### The three questions every learning-curve reading should answer:
1. **Where do the curves converge?** — tells you the irreducible error floor
2. **How large is the gap at max training size?** — tells you variance level
3. **Is the validation curve still falling?** — tells you if more data will help

## Self-Check Questions

Answer each question on your own, then expand the answer.

---

**Q1.** Your model's learning curve shows: training MSE = 0.05, validation MSE = 0.08 at n = 10 000, and both curves are flat. What does this tell you? What should you try?

<details>
<summary>Show answer</summary>

**Diagnosis:** Both curves are low and flat, and they have converged to a small gap (0.03). This is a **well-fit model** — there is no severe bias or variance problem.

**What to try:**
- The model has essentially extracted what it can from this feature set.
- Collecting more data will give only marginal improvements because the validation curve has already plateaued.
- To improve further, you need to **engineer better features**, add domain knowledge, or try a more expressive model class (e.g., a gradient-boosted tree or neural network).
- The remaining 0.03 gap is mostly irreducible noise — don't over-optimise it away.

</details>

---

**Q2.** Training MSE = 0.10, validation MSE = 0.45 at n = 5 000, with a downward trend in validation MSE. Should you get more data or simplify the model?

<details>
<summary>Show answer</summary>

**Diagnosis:** The large gap (0.35) signals **high variance (overfitting)**. The validation curve is *still falling*, which means more data is actively helping.

**Answer: Get more data.** Here's why:
- A downward trend in validation MSE is the key signal — the model has not yet hit its data-capacity limit.
- More training data will reduce the overfitting gap.

**Also consider simultaneously:**
- Add regularisation (Ridge/Lasso/ElasticNet) to tighten the gap faster.
- If data collection is very expensive, simplifying the model is the budget-friendly alternative.
- Do not simplify aggressively before collecting more data — a simpler model might close the gap but raise the floor (introduce bias).

</details>

---

**Q3.** You double training data from 1 000 → 2 000. Validation error drops from 0.35 → 0.28. From 2 000 → 4 000 it drops from 0.28 → 0.26. Is more data still helping? What's the diminishing-returns pattern?

<details>
<summary>Show answer</summary>

**Is more data helping?** Yes, but with **strong diminishing returns**.

| Data range | Drop in val error | Gain per 1 000 samples |
|---|---|---|
| 1 000 → 2 000 | 0.07 | 0.070 |
| 2 000 → 4 000 | 0.02 | 0.010 |

**The pattern:** Validation error follows a roughly **power-law** decay with sample size: $\text{val\_err}(n) \approx a \cdot n^{-b} + c$.
- Early data points give large gains (high slope).
- Later data points give diminishing gains (slope flattening).
- Eventually the curve approaches an asymptote $c$ — the irreducible error — and no amount of data helps further.

**Practical decision:** Use the power-law extrapolation (as in Section 7) to estimate how many samples are needed to reach your target error. If the cost of 10× more data only buys 0.01 improvement, it may be better to invest in **better features or a better model**.

</details>

In [ ]:
# ── Bonus: interactive diminishing-returns illustration ──────────────────────
# Demonstrates the power-law shape of validation error vs sample size

np.random.seed(42)
n_values   = np.array([100, 200, 500, 1000, 2000, 4000, 8000, 16000])
# Simulate: val_err ~ 3 * n^(-0.4) + 0.05
noise_sim  = np.random.normal(0, 0.01, len(n_values))
val_errors = 3.0 * np.power(n_values.astype(float), -0.4) + 0.05 + noise_sim

# Fit the power law to the simulated data
popt2, _ = curve_fit(power_law, n_values.astype(float), val_errors,
                     p0=[3, 0.4, 0.05], maxfev=5000)

n_ext   = np.logspace(np.log10(100), np.log10(50000), 200)
fit_ext = power_law(n_ext, *popt2)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, xscale in zip(axes, ['linear', 'log']):
    ax.plot(n_values, val_errors, 'o', color='tomato', ms=8, label='Observed val MSE')
    ax.plot(n_ext,    fit_ext,    '-', color='steelblue', lw=2, label='Power-law fit')
    ax.axhline(popt2[2], color='green', ls='--', lw=1.5,
               label=f'Asymptote (irred. error) ≈ {popt2[2]:.3f}')
    ax.set_xscale(xscale)
    ax.set_xlabel('Training samples' + (' (log scale)' if xscale == 'log' else ''))
    ax.set_ylabel('Validation MSE')
    ax.set_title(f'Diminishing Returns — {xscale.title()} Scale')
    ax.legend(fontsize=9)
    ax.set_ylim(bottom=0)

plt.suptitle('Power-Law Decay of Validation Error vs Training Size',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Fitted power-law: val_err ≈ {popt2[0]:.2f} × n^(-{popt2[1]:.2f}) + {popt2[2]:.3f}")
print(f"Irreducible error floor ≈ {popt2[2]:.3f}")